# Error analysis: what's driving MAPE, and why comps and geo didn't help
Evidence for `06-plan.md`. Every finding has an ID (E1–E12) that the plan refers to.

CRISP-DM: this is **Evaluation** (task 5.1, assess results) feeding back into **Data understanding** and **Data preparation**. It asks *why* 05's feature groups did what they did before more features get built.

#### Test design (written before running)
- **Steps 1–4, one split:** train = Jan 2024 – Feb 2026, validation = Mar 2026. Same split as 03, 04 and `fe_metrics.csv`. Model: `rf_leaf5_log` (100 trees, target-encoded City/County/ZIP), the test model from 05.
- **Step 5, rolling 5-fold CV** (Nov 2025 – Mar 2026), same design as 05, for the four comparisons the plan depends on. A change counts only if its paired-bootstrap 95% interval excludes 0 **and** at least 4 of 5 months agree.
- **Fit on train only:** fills, encoders and the comps pool come from train, or from each fold's train rows.
- **Test (Apr 2026) is not read.**
- **ListPrice (E3) is a diagnostic only.** It's read for the validation month and is never a model input: it's leakage.

#### Setup

In [1]:
import os
import time

import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import BallTree
from sklearn.pipeline import Pipeline

from idx_pipeline import (CAT_COLS, DATA_DIR, EARTH_KM, RAW_DIR, SEED, TARGET, DatasetAImputer, add_comps,
                          add_similar_comps, build_comp_pool, build_preprocessor, error_by, fill_coords,
                          load_split, metrics, rolling_folds)

print("sklearn", sklearn.__version__)
MODEL_DIR = os.path.join(DATA_DIR, "models")
description = pd.read_csv(os.path.join(DATA_DIR, "dataset_description.csv"))
BASE_NUM = [c for c in description.loc[description["role"].isin(["model_feature", "missing_flag"]), "column"]
            if c not in CAT_COLS]  # Dataset A, same as 04 and 05

PLAIN_COMPS = ["comp_ppsf_median", "comp_ppsf_mean", "comp_km_kth", "comp_count", "sales_within_1km",
               "comp_implied_value"]  # 05's comps: 10 nearest sales
SIM_COMPS = ["sim_ppsf", "sim_ppsf_iqr", "sim_count", "sim_value"]  # similar homes only


def fit_predict(train, val, extra=(), max_features=1.0):
    # rf_leaf5_log from 04/05, 100 trees. Returns validation predictions in dollars.
    num = BASE_NUM + list(extra)
    model = TransformedTargetRegressor(
        regressor=Pipeline([("pre", build_preprocessor("tree_te", num, CAT_COLS)),
                            ("model", RandomForestRegressor(n_estimators=100, min_samples_leaf=5,
                                                            max_features=max_features, n_jobs=-1,
                                                            random_state=SEED))]),
        func=np.log, inverse_func=np.exp)
    model.fit(train[num + CAT_COLS], train[TARGET].to_numpy())
    return model.predict(val[num + CAT_COLS])

sklearn 1.9.1


#### Step 1: Base model on the March 2026 split

In [2]:
start = time.time()
train, val = load_split("train"), load_split("val")
y = val[TARGET].to_numpy()
pred = fit_predict(train, val)
val = val.assign(pred=pred, ape=np.abs(pred - y) / y,
                 log_err=np.log(pred / y))  # log_err > 0: overpredicted
print(metrics(y, pred), f"{time.time() - start:.0f}s")

{'MdAPE_%': np.float64(7.78), 'MAPE_%': np.float64(11.38), 'R2': 0.8872, 'MAE_$': 155370, 'within_10%': np.float64(60.1)} 116s


#### Step 2: Where the error comes from
**E9: how concentrated is MAPE?** MAPE is a mean, so a few very wrong homes can dominate it.

In [3]:
s = np.sort(val["ape"].to_numpy())[::-1]
pd.DataFrame({f"worst {q:.0%}": {"share of MAPE total": f"{s[:int(q * len(s))].sum() / s.sum():.1%}",
                                 "MAPE without them": f"{s[int(q * len(s)):].mean():.2%}"}
              for q in [0.01, 0.05, 0.10]})

,worst 1%,worst 5%,worst 10%
share of MAPE total,7.1%,22.2%,35.0%
MAPE without them,10.67%,9.32%,8.22%


**E7: pull toward the average.** Signed bias = median log error (positive = overpredicted). Grouped two ways:
- by *actual* price, which is what the price-band tables in 04/05 show
- by *predicted* price, which is what a user of the model sees and what could be corrected after the fact

In [4]:
def band_table(by):
    bands = pd.qcut(val[by], 10, labels=[f"D{i}" for i in range(1, 11)])
    g = val.groupby(bands, observed=True)
    return pd.DataFrame({"median_price": g[TARGET].median().round(-3), "MdAPE_%": (100 * g["ape"].median()).round(2),
                         "MAPE_%": (100 * g["ape"].mean()).round(2), "bias_%": (100 * g["log_err"].median()).round(1)})


pd.concat({"by actual price": band_table(TARGET), "by predicted price": band_table("pred")}, axis=1)

by actual price                       by predicted price                 \
       median_price MdAPE_% MAPE_% bias_%       median_price MdAPE_% MAPE_%   
D1         370000.0    7.77  13.92    3.8           377000.0    7.19  11.55   
D2         510000.0    6.23  10.26    0.3           518000.0    5.94   9.19   
D3         625000.0    5.53   8.66   -0.1           630000.0    5.56   8.45   
D4         730000.0    6.04   8.78   -0.0           740000.0    6.06   8.87   
D5         830000.0    6.18   8.77   -0.4           844000.0    6.33   8.88   
D6         960000.0    7.26  10.01   -1.1           960000.0    8.22  11.14   
D7        1177000.0    8.68  11.32   -1.7          1180000.0    8.84  11.91   
D8        1433000.0    9.34  12.30   -2.4          1433000.0    9.43  12.33   
D9        1850000.0   10.26  13.05   -5.6          1835000.0   10.40  14.29   
D10       3022000.0   13.75  16.70  -10.6          2974000.0   12.69  17.13   

            
    bias_%  
D1    -0.2  
D2    -1.1  
D3    -1.1  
D4    -1.3  
D5    -0.6  
D6    -0.2  
D7    -1.4  
D8    -1.8  
D9    -2.0  
D10   -3.3

**E2: is the error about the home or about the location?** For each validation home: how far its price/sqft is from its ZIP's median price/sqft in train. If the model were missing location information, this would barely matter. If it predicts "a typical home here", it matters a lot.

In [5]:
zip_ppsf = train.groupby("PostalCode")["price_per_sqft"].median()
val["vs_zip"] = np.log(val["price_per_sqft"] / val["PostalCode"].map(zip_ppsf))
ok = val["vs_zip"].notna()
print("corr(model log error, log(price/sqft vs ZIP median)) =",
      round(np.corrcoef(val.loc[ok, "log_err"], val.loc[ok, "vs_zip"])[0, 1], 3))
dec = pd.qcut(val["vs_zip"], 10, labels=["furthest below ZIP"] + [f"D{i}" for i in range(2, 10)] + ["furthest above ZIP"])
g = val.groupby(dec, observed=True)
pd.DataFrame({"n": g.size(), "MdAPE_%": (100 * g["ape"].median()).round(1), "bias_%": (100 * g["log_err"].median()).round(1)}).T

corr(model log error, log(price/sqft vs ZIP median)) = -0.68


vs_zip,furthest below ZIP,D2,D3,D4,D5,D6,D7,D8,D9,furthest above ZIP
n,1141.0,1140.0,1140.0,1140.0,1140.0,1140.0,1140.0,1140.0,1140.0,1141.0
MdAPE_%,18.9,9.0,7.1,5.9,5.5,5.6,6.2,7.0,9.0,14.1
bias_%,17.3,7.6,3.9,1.6,-0.4,-2.6,-4.1,-5.9,-8.4,-14.9


**E1: is there location signal left in the errors?** If neighbors are wrong in the same direction, location features could still help. Compare each home's log error with the median error of its 10 nearest validation homes (real coordinates only).

In [6]:
real = val[val["latlon_missing"] == 0]
xy = np.radians(real[["Latitude", "Longitude"]].to_numpy())
dist, idx = BallTree(xy, metric="haversine").query(xy, k=11)  # first neighbor is the home itself
neighbor_err = np.median(real["log_err"].to_numpy()[idx[:, 1:]], axis=1)
print("corr(own error, neighbors' median error) =", round(np.corrcoef(real["log_err"], neighbor_err)[0, 1], 3))
print("median km to the 10th neighbor:", round(np.median(dist[:, -1]) * EARTH_KM, 2))

corr(own error, neighbors' median error) = 0.16
median km to the 10th neighbor: 2.06


**E8: market trend.** A rising market that the model doesn't know about shows up as underprediction.

In [7]:
print("overall median bias:", f"{100 * val['log_err'].median():.2f}%")
week = val["CloseDate"].dt.to_period("W").astype(str)
val.groupby(week).agg(n=("ape", "size"), bias_pct=("log_err", lambda e: round(100 * e.median(), 1))).T

overall median bias: -1.14%


CloseDate,2026-02-23/2026-03-01,2026-03-02/2026-03-08,2026-03-09/2026-03-15,2026-03-16/2026-03-22,2026-03-23/2026-03-29,2026-03-30/2026-04-05
n,2.0,2429.0,2435.0,2595.0,2820.0,1131.0
bias_pct,5.2,-1.3,-1.1,-0.9,-0.9,-1.7


**E3 (diagnostic only, never a feature): what does the model not know?** ListPrice is set by an agent who has seen the house. If it explains the model's errors, the missing information is the home's condition and quality. ListPrice is leakage: it encodes the answer and doesn't exist for homes that aren't for sale. Only the validation month's file is read.

In [8]:
raw_val = pd.read_csv(os.path.join(RAW_DIR, "CRMLSSold202603.csv"), usecols=["ListingId", "ListPrice", "OriginalListPrice"],
                      dtype={"ListingId": str}).drop_duplicates("ListingId", keep="last")
lp = val[["ListingId", "log_err", "pred", TARGET]].astype({"ListingId": str}).merge(raw_val, on="ListingId", how="left")
for c in ["ListPrice", "OriginalListPrice"]:
    ok = lp[c] > 0
    print(f"{c}: coverage {ok.mean():.1%}, corr(model log error, log(pred / {c})) =",
          round(np.corrcoef(lp.loc[ok, "log_err"], np.log(lp.loc[ok, "pred"] / lp.loc[ok, c]))[0, 1], 3),
          f"| {c} alone MdAPE {100 * (np.abs(lp.loc[ok, c] - lp.loc[ok, TARGET]) / lp.loc[ok, TARGET]).median():.1f}%")
del raw_val, lp  # not used again

ListPrice: coverage 100.0%, corr(model log error, log(pred / ListPrice)) = 0.89 | ListPrice alone MdAPE 2.2%
OriginalListPrice: coverage 99.7%, corr(model log error, log(pred / OriginalListPrice)) = 0.572 | OriginalListPrice alone MdAPE 3.1%


#### Step 3: Why comps hurt
Build 05's comps (10 nearest sales) and similar comps (10 nearest with LivingArea ±25% and YearBuilt ±15). Both use a train-only pool, strictly earlier sales, and 180 days.

In [9]:
start = time.time()
imputer = DatasetAImputer().fit(train)
lat, lon = fill_coords(train, imputer)
vlat, vlon = fill_coords(val, imputer)
pool = build_comp_pool(train, lat, lon)
train_c = add_similar_comps(add_comps(train, lat, lon, pool), lat, lon, pool)
val_c = add_similar_comps(add_comps(val, vlat, vlon, pool), vlat, vlon, pool)
print(f"{time.time() - start:.0f}s")

116s


**E6: do comps look different in train and validation?** If they did, that mismatch could explain the drop.

In [10]:
def comp_profile(d):
    ok = d["comp_ppsf_median"].notna() & d["price_per_sqft"].notna()
    return {"has comps": f"{(d['comp_count'] > 0).mean():.1%}", "median km to 10th comp": round(d["comp_km_kth"].median(), 2),
            "corr(log comp ppsf, log own ppsf)": round(np.corrcoef(np.log(d.loc[ok, "comp_ppsf_median"]),
                                                                   np.log(d.loc[ok, "price_per_sqft"]))[0, 1], 3),
            "has similar comps": f"{d['sim_ppsf'].notna().mean():.1%}"}


display(pd.DataFrame({"train": comp_profile(train_c), "val": comp_profile(val_c)}))
print("train homes with any comp, by month (early 2024 has no earlier sales):")
train_c.groupby("source_month")["comp_count"].apply(lambda c: f"{(c > 0).mean():.0%}").head(4).to_frame().T

,train,val
has comps,95.7%,100.0%
median km to 10th comp,0.86,0.83
"corr(log comp ppsf, log own ppsf)",0.894,0.906
has similar comps,90.2%,97.4%


train homes with any comp, by month (early 2024 has no earlier sales):


source_month,2024-01,2024-02,2024-03,2024-04
comp_count,72%,84%,85%,86%


**E4: are the comps good comps?** Two tests on validation:
- *within-ZIP correlation*: does the comp price/sqft's distance from the ZIP median track the home's own? This is the information a comp adds beyond the ZIP.
- *comp value alone*: comp price/sqft × LivingArea as a prediction, with no model.

In [11]:
v = val_c.assign(zip_ppsf=val_c["PostalCode"].map(zip_ppsf))
rows = {}
for name, ppsf, value in [("05 comps (10 nearest)", "comp_ppsf_median", "comp_implied_value"),
                          ("similar comps", "sim_ppsf", "sim_value")]:
    ok = v[ppsf].notna() & v["zip_ppsf"].notna() & v["price_per_sqft"].notna()
    rows[name] = {"coverage": f"{v[ppsf].notna().mean():.1%}",
                  "within-ZIP corr": round(np.corrcoef(np.log(v.loc[ok, "price_per_sqft"] / v.loc[ok, "zip_ppsf"]),
                                                       np.log(v.loc[ok, ppsf] / v.loc[ok, "zip_ppsf"]))[0, 1], 3),
                  "MdAPE alone %": round(100 * (np.abs(v.loc[ok, value] - v.loc[ok, TARGET]) / v.loc[ok, TARGET]).median(), 2)}
pd.DataFrame(rows).T

,coverage,within-ZIP corr,MdAPE alone %
05 comps (10 nearest),100.0%,0.328,13.23
similar comps,97.4%,0.586,9.48


**E5: do they help the forest?** Each row is the base model plus one comp set, with and without `max_features=0.33` (each split sees a random third of the columns, so a dominant noisy column can't take over every tree). About 1 minute per fit.

In [12]:
top = y >= np.quantile(y, 0.9)
configs = {"base": ((), 1.0), "base, mf=0.33": ((), 0.33),
           "+ 05 comps": (PLAIN_COMPS, 1.0), "+ 05 comps, mf=0.33": (PLAIN_COMPS, 0.33),
           "+ similar comps": (SIM_COMPS, 1.0), "+ similar comps, mf=0.33": (SIM_COMPS, 0.33)}
e5 = {}
for name, (extra, mf) in configs.items():
    p = pred if name == "base" else fit_predict(train_c, val_c, extra, mf)
    m = metrics(y, p)
    e5[name] = {"MdAPE_%": m["MdAPE_%"], "MAPE_%": m["MAPE_%"],
                "top10%_MdAPE": round(100 * np.median(np.abs(p[top] - y[top]) / y[top]), 2)}
    print(name, e5[name])
e5 = pd.DataFrame(e5).T
e5.to_csv(os.path.join(MODEL_DIR, "06_comps_single_split.csv"))
e5

base {'MdAPE_%': np.float64(7.78), 'MAPE_%': np.float64(11.38), 'top10%_MdAPE': np.float64(13.73)}
base, mf=0.33 {'MdAPE_%': np.float64(7.73), 'MAPE_%': np.float64(11.3), 'top10%_MdAPE': np.float64(13.86)}
+ 05 comps {'MdAPE_%': np.float64(7.98), 'MAPE_%': np.float64(11.52), 'top10%_MdAPE': np.float64(13.24)}
+ 05 comps, mf=0.33 {'MdAPE_%': np.float64(7.94), 'MAPE_%': np.float64(11.36), 'top10%_MdAPE': np.float64(13.35)}
+ similar comps {'MdAPE_%': np.float64(7.93), 'MAPE_%': np.float64(11.65), 'top10%_MdAPE': np.float64(13.32)}
+ similar comps, mf=0.33 {'MdAPE_%': np.float64(7.76), 'MAPE_%': np.float64(11.43), 'top10%_MdAPE': np.float64(13.35)}


,MdAPE_%,MAPE_%,top10%_MdAPE
base,7.78,11.38,13.73
"base, mf=0.33",7.73,11.30,13.86
+ 05 comps,7.98,11.52,13.24
"+ 05 comps, mf=0.33",7.94,11.36,13.35
+ similar comps,7.93,11.65,13.32
"+ similar comps, mf=0.33",7.76,11.43,13.35


#### Step 4: Who the errors fall on (ethics check)
**E10: error by county** (10 largest). Signed bias shows *who* is over- or underpredicted, not just by how much. This is an error-by-area check, **not** a fairness audit: there is no demographic data in the set.

In [13]:
top_counties = val["CountyOrParish"].value_counts().index[:10]
sub = val[val["CountyOrParish"].isin(top_counties)]
county = error_by(sub, sub["pred"].to_numpy(), "CountyOrParish")[["n", "MdAPE_%", "MAPE_%"]]
county["bias_%"] = (100 * sub.groupby("CountyOrParish")["log_err"].median()).round(1)
county["median_price"] = sub.groupby("CountyOrParish")[TARGET].median().round(-3)
county.sort_values("MdAPE_%")

,n,MdAPE_%,MAPE_%,bias_%,median_price
San Bernardino,1196.0,6.06,9.59,-1.5,549000.0
Riverside,1917.0,6.09,10.26,-0.2,640000.0
Ventura,369.0,7.24,10.48,-0.9,965000.0
Contra Costa,538.0,7.26,10.97,1.5,875000.0
San Diego,1273.0,7.41,10.88,-1.2,1080000.0
Santa Clara,337.0,8.32,10.69,-3.8,2000000.0
Orange,1099.0,8.40,10.70,-3.1,1420000.0
Los Angeles,2785.0,8.97,12.49,-1.1,1015000.0
Alameda,491.0,9.84,13.27,-2.4,1375000.0
San Luis Obispo,202.0,10.51,13.86,-3.0,949000.0


**E11: how many homes have an earlier sale?** This limits both repeat-sales features and any per-home sequence model ("predict the next close price").

In [14]:
last = train.sort_values("CloseDate").groupby("property_id").agg(prev_price=(TARGET, "last"), prev_date=("CloseDate", "last"))
r = val.join(last, on="property_id", how="inner")
r = r[r["prev_date"] < r["CloseDate"]]
print(f"{len(r)} of {len(val)} validation homes ({len(r) / len(val):.1%}) have an earlier sale in train;",
      f"median {((r['CloseDate'] - r['prev_date']).dt.days / 30.4).median():.0f} months earlier")
print(f"model MdAPE on them {100 * r['ape'].median():.1f}%, previous price alone (not time-adjusted)",
      f"{100 * (np.abs(r['prev_price'] - r[TARGET]) / r[TARGET]).median():.1f}%")

513 of 11412 validation homes (4.5%) have an earlier sale in train; median 13 months earlier
model MdAPE on them 9.7%, previous price alone (not time-adjusted) 17.3%


**E12: regularization**, from 04's saved results: the overfit gap vs validation MdAPE.

In [15]:
pd.read_csv(os.path.join(MODEL_DIR, "trees_metrics.csv"), index_col=0)[["val_MdAPE_%", "val_MAPE_%", "train_MdAPE_%", "gap_MdAPE"]].head(4)

,val_MdAPE_%,val_MAPE_%,train_MdAPE_%,gap_MdAPE
rf_log,7.69,11.30,2.69,5.00
rf,7.72,11.98,2.72,5.00
rf_leaf5_log,7.73,11.37,4.75,2.98
rf_leaf5,7.77,11.99,4.82,2.95


#### Step 5: Rolling cross-validation of the comp comparisons
**Why CV:** steps 1–4 use March 2026 only. Differences like base vs `mf=0.33` (about 0.05 pts) are smaller than the month-to-month swing, and ~15 feature sets have already been compared on March, so choosing on March alone would make the winner look better than it is.

**Why rolling instead of random folds:** random folds train on future sales to predict past ones. That leaks the market trend (E8) and lets comps see later prices. Here each fold trains on every month before its validation month, and fills, encoders and the comps pool are rebuilt from those rows only. About 30–40 minutes.

In [16]:
VAL_MONTHS = ["2025-11", "2025-12", "2026-01", "2026-02", "2026-03"]
dev = pd.concat([train, val[train.columns]], ignore_index=True)  # train + val, no test
del train_c, val_c

folds = {}
start = time.time()
for month, tr, va in rolling_folds(dev, VAL_MONTHS):
    assert tr["source_month"].max() < month  # train is always earlier than validation
    imp = DatasetAImputer().fit(tr)
    tlat, tlon = fill_coords(tr, imp)
    vlat_f, vlon_f = fill_coords(va, imp)
    fold_pool = build_comp_pool(tr, tlat, tlon)
    folds[month] = (add_similar_comps(add_comps(tr, tlat, tlon, fold_pool), tlat, tlon, fold_pool),
                    add_similar_comps(add_comps(va, vlat_f, vlon_f, fold_pool), vlat_f, vlon_f, fold_pool))
    print(month, "train", len(tr), "val", len(va), f"{time.time() - start:.0f}s")
y_cv = np.concatenate([folds[m][1][TARGET].to_numpy() for m in VAL_MONTHS])

2025-11 train 246137 val 9596 91s
2025-12 train 255733 val 10317 193s
2026-01 train 266050 val 7370 292s
2026-02 train 273420 val 8859 400s
2026-03 train 282279 val 11412 512s


In [17]:
CV_CONFIGS = {"base": ((), 1.0), "base, mf=0.33": ((), 0.33),
              "+ 05 comps": (PLAIN_COMPS, 1.0), "+ similar comps, mf=0.33": (SIM_COMPS, 0.33)}
cv_apes, cv_rows = {}, {}
for name, (extra, mf) in CV_CONFIGS.items():
    start, preds, per_fold = time.time(), [], {}
    for month in VAL_MONTHS:
        tr, va = folds[month]
        preds.append(fit_predict(tr, va, extra, mf))
        per_fold[f"MdAPE_{month}"] = metrics(va[TARGET].to_numpy(), preds[-1])["MdAPE_%"]
    p = np.concatenate(preds)
    assert len(p) == len(y_cv)  # every config scored on the same homes
    cv_apes[name] = np.abs(p - y_cv) / y_cv
    m = metrics(y_cv, p)
    cv_rows[name] = {"cv_MdAPE_%": m["MdAPE_%"], "cv_MAPE_%": m["MAPE_%"], **per_fold}
    print(name, m["MdAPE_%"], m["MAPE_%"], f"{time.time() - start:.0f}s")

base 7.85 11.68 223s
base, mf=0.33 7.81 11.63 79s
+ 05 comps 8.0 11.75 381s
+ similar comps, mf=0.33 7.8 11.68 132s


Paired bootstrap against `base`: the same resampled homes for both models. Negative = better. **Helps / hurts** only if the 95% interval excludes 0 and at least 4 of 5 months agree.

In [18]:
def bootstrap_change(new, ref="base", n=1000):
    rng = np.random.default_rng(SEED)
    idx = rng.integers(0, len(y_cv), (n, len(y_cv)))
    a, b = cv_apes[new][idx], cv_apes[ref][idx]
    med = 100 * (np.median(a, axis=1) - np.median(b, axis=1))
    mean = 100 * (a.mean(axis=1) - b.mean(axis=1))
    better = sum(cv_rows[new][f"MdAPE_{m}"] < cv_rows[ref][f"MdAPE_{m}"] for m in VAL_MONTHS)
    lo, hi = np.percentile(med, [2.5, 97.5])
    verdict = ("helps" if hi < 0 and better >= 4 else "hurts" if lo > 0 and better <= 1 else "no clear effect")
    return {"MdAPE_change": round(100 * (np.median(cv_apes[new]) - np.median(cv_apes[ref])), 2),
            "ci_low": round(lo, 2), "ci_high": round(hi, 2), "months_better": better,
            "MAPE_change": round(100 * (cv_apes[new].mean() - cv_apes[ref].mean()), 2),
            "mape_ci": f"[{np.percentile(mean, 2.5):.2f}, {np.percentile(mean, 97.5):.2f}]", "verdict": verdict}


cv = pd.DataFrame(cv_rows).T
change = pd.DataFrame({n: bootstrap_change(n) for n in CV_CONFIGS if n != "base"}).T
cv_out = cv.join(change)
cv_out.to_csv(os.path.join(MODEL_DIR, "06_cv_comps.csv"))
cv_out

,cv_MdAPE_%,cv_MAPE_%,MdAPE_2025-11,MdAPE_2025-12,MdAPE_2026-01,MdAPE_2026-02,MdAPE_2026-03,MdAPE_change,ci_low,ci_high,months_better,MAPE_change,mape_ci,verdict
base,7.85,11.68,7.87,7.82,7.74,8.03,7.78,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"base, mf=0.33",7.81,11.63,7.78,7.81,7.77,8.02,7.73,-0.04,-0.09,0.03,4,-0.05,"[-0.08, -0.01]",no clear effect
+ 05 comps,8.00,11.75,7.96,7.89,7.99,8.21,7.98,0.15,0.07,0.23,0,0.08,"[0.01, 0.14]",hurts
"+ similar comps, mf=0.33",7.80,11.68,7.71,7.78,7.78,7.96,7.76,-0.05,-0.14,0.03,4,0.0,"[-0.06, 0.06]",no clear effect


#### Findings
*Written after the run; see `06-plan.md` for how each finding feeds the next steps.*